# Lab 02-05: Re-ranking

**Module:** 02 -- Data Preparation (14% of exam)  
**UI Sections:** Catalog  
**Estimated Time:** 30--40 minutes  
**Difficulty:** Intermediate

---

## What and Why

Vector search is fast but approximate. It uses Approximate Nearest Neighbor (ANN) algorithms
to find documents with embeddings close to the query embedding. This is great for speed but
sometimes misses nuance -- two texts can have similar embeddings without being truly relevant
to the same question.

A **re-ranker** takes the rough candidates from vector search and re-scores them with a more
accurate (but slower) model. This gives you **two-stage retrieval**:

1. **Stage 1 (fast, broad)**: Vector search retrieves many candidates (e.g., 20-50)
2. **Stage 2 (slow, precise)**: Re-ranker scores each candidate against the query and
   returns only the top K (e.g., 3-5)

---

## Mental Model

> **Re-ranking is like a talent show.**
>
> **Round 1 (vector search)**: Judges quickly screen 50 performers and pick 20 with
> potential. This round is fast -- they watch 30 seconds of each act. Some genuinely
> talented performers slip through, but some mediocre ones do too.
>
> **Round 2 (re-ranker)**: A panel of expert judges carefully evaluates those 20 and
> picks the top 5. They watch each full act, ask questions, and deliberate. The second
> round is slower but much more accurate.
>
> You would never have the expert panel watch all 50 acts (too slow). And you would
> never let the quick screening be the final decision (too inaccurate). The two stages
> complement each other.

---

## Exam Alert

**Re-ranking adds latency.** The exam will test whether you understand the trade-off:
re-ranking is justified when **precision matters more than speed** (legal, medical,
compliance use cases). For low-latency chat applications where "good enough" retrieval
is acceptable, re-ranking may not be worth the extra time.

---

## Learning Objectives

By the end of this lab you will be able to:

1. Explain the two-stage retrieval architecture and why each stage exists
2. Simulate a vector search returning candidates with similarity scores
3. Implement a cross-encoder re-ranker that scores query-document pairs
4. Compare rankings before and after re-ranking
5. Decide when re-ranking is worth the latency trade-off

---

## Exam Objectives Covered

| Objective | Tested Here |
|-----------|-------------|
| Two-stage retrieval architecture | Step 1 |
| Re-ranking concepts and trade-offs | Steps 3--5 |
| Cross-encoder vs bi-encoder | Steps 1, 3 |
| Latency vs precision trade-off | Step 5 |

---

## Step 1: Understand the Two-Stage Retrieval Architecture

### Bi-Encoder vs Cross-Encoder

The key to understanding re-ranking is the difference between these two model architectures:

**Bi-Encoder (used in Stage 1 -- Vector Search)**
- Encodes the query and each document **independently** into separate vectors
- Similarity = cosine distance between the two vectors
- FAST: you pre-compute document vectors once, then just compute the query vector at search time
- APPROXIMATE: because query and document never "see" each other during encoding

**Cross-Encoder (used in Stage 2 -- Re-Ranker)**
- Takes the query AND document **together** as a single input
- Outputs a single relevance score (not a vector)
- SLOW: must run the model once for every (query, document) pair
- ACCURATE: because the model can attend to interactions between query and document

### The Architecture

```
Query
  |                                                     
  v                                                     
[Bi-Encoder]  -->  Vector Search (ANN)  -->  Top 20 candidates
                                                  |
                                                  v
                                          [Cross-Encoder Re-Ranker]
                                                  |
                                                  v
                                           Top 5 final results
```

Think of it as a funnel: wide at the top (many candidates), narrow at the bottom (few, high-quality results).

In [ ]:
# ---------------------------------------------------------------------------
# Visualize the two-stage architecture with concrete numbers
# ---------------------------------------------------------------------------
print("Two-Stage Retrieval Architecture")
print("=" * 55)
print()
print("                    FULL CORPUS")
print("               (100,000 documents)")
print("                       |")
print("                       v")
print("  +----------------------------------------+")
print("  |  STAGE 1: Vector Search (Bi-Encoder)   |")
print("  |  Speed: ~50ms for 100K docs             |")
print("  |  Method: ANN (approximate)              |")
print("  |  Output: Top 20 candidates              |")
print("  +----------------------------------------+")
print("                       |")
print("                  20 candidates")
print("                       |")
print("                       v")
print("  +----------------------------------------+")
print("  |  STAGE 2: Re-Ranker (Cross-Encoder)    |")
print("  |  Speed: ~200ms for 20 candidates        |")
print("  |  Method: Exact scoring per pair          |")
print("  |  Output: Top 5 final results             |")
print("  +----------------------------------------+")
print("                       |")
print("                   5 results")
print("                       |")
print("                       v")
print("                    TO LLM")
print()
print("Total latency: ~250ms (vs ~50ms without re-ranking)")
print("Quality improvement: significant for nuanced queries")

---

## Step 2: Simulate Vector Search Returning 20 Candidates

Let us simulate what Stage 1 returns. In a real system, this would be the output of
Databricks Vector Search. Each candidate has a document ID, text, and a cosine similarity
score from the bi-encoder.

Notice: the bi-encoder scores are sometimes misleading. A document about "model training"
might score high for a query about "model serving" because the embeddings for "model" are
similar -- even though the topics are quite different.

In [ ]:
# ---------------------------------------------------------------------------
# Simulate vector search results for a query about model deployment
# ---------------------------------------------------------------------------
query = "How do I deploy and serve a machine learning model as a REST API?"

# These are the 20 candidates that vector search returned, with cosine similarity scores.
# Some are truly relevant, some are false positives (high similarity but wrong topic).
vector_search_results = [
    {"doc_id": "D01", "text": "Model Serving creates REST API endpoints for ML models with auto-scaling.", "bi_score": 0.94, "truly_relevant": True},
    {"doc_id": "D02", "text": "MLflow model registry tracks model versions and deployment stages.", "bi_score": 0.91, "truly_relevant": True},
    {"doc_id": "D03", "text": "Training deep learning models requires GPU clusters and distributed computing.", "bi_score": 0.89, "truly_relevant": False},
    {"doc_id": "D04", "text": "REST APIs should follow proper authentication and rate limiting practices.", "bi_score": 0.87, "truly_relevant": False},
    {"doc_id": "D05", "text": "Serving endpoints support A/B testing with traffic splitting between model versions.", "bi_score": 0.86, "truly_relevant": True},
    {"doc_id": "D06", "text": "Machine learning pipelines automate feature engineering and model training.", "bi_score": 0.85, "truly_relevant": False},
    {"doc_id": "D07", "text": "Deploy models using the Databricks serving endpoint UI or REST API.", "bi_score": 0.84, "truly_relevant": True},
    {"doc_id": "D08", "text": "Kubernetes pods can host containerized model serving infrastructure.", "bi_score": 0.83, "truly_relevant": False},
    {"doc_id": "D09", "text": "Foundation Model APIs provide access to DBRX, Llama, and Mixtral on Databricks.", "bi_score": 0.82, "truly_relevant": True},
    {"doc_id": "D10", "text": "Data pipelines transform raw data into feature tables for ML training.", "bi_score": 0.80, "truly_relevant": False},
    {"doc_id": "D11", "text": "Model serving endpoints scale to zero when not in use, reducing costs.", "bi_score": 0.79, "truly_relevant": True},
    {"doc_id": "D12", "text": "REST API design best practices include versioning and proper error codes.", "bi_score": 0.78, "truly_relevant": False},
    {"doc_id": "D13", "text": "External models like OpenAI GPT-4 can be accessed through AI Gateway.", "bi_score": 0.77, "truly_relevant": True},
    {"doc_id": "D14", "text": "Hyperparameter tuning with Optuna optimizes model performance.", "bi_score": 0.76, "truly_relevant": False},
    {"doc_id": "D15", "text": "Model monitoring tracks prediction drift and data quality over time.", "bi_score": 0.75, "truly_relevant": False},
    {"doc_id": "D16", "text": "Batch inference processes large datasets without real-time latency constraints.", "bi_score": 0.74, "truly_relevant": False},
    {"doc_id": "D17", "text": "Feature Store manages features for consistent training and serving.", "bi_score": 0.73, "truly_relevant": False},
    {"doc_id": "D18", "text": "Serving endpoint logs can be queried for debugging and auditing.", "bi_score": 0.72, "truly_relevant": True},
    {"doc_id": "D19", "text": "Delta Lake provides ACID transactions for reliable data storage.", "bi_score": 0.71, "truly_relevant": False},
    {"doc_id": "D20", "text": "Spark clusters can be configured with autoscaling for variable workloads.", "bi_score": 0.70, "truly_relevant": False},
]

print(f"Query: \"{query}\"")
print(f"\nVector Search returned {len(vector_search_results)} candidates.")
print(f"Truly relevant: {sum(1 for d in vector_search_results if d['truly_relevant'])}")
print(f"False positives: {sum(1 for d in vector_search_results if not d['truly_relevant'])}")
print()
print(f"{'Rank':<6} {'DocID':<6} {'BiScore':<9} {'Relevant':<10} Text")
print("-" * 90)
for i, doc in enumerate(vector_search_results):
    marker = "YES" if doc["truly_relevant"] else "---"
    print(f"{i+1:<6} {doc['doc_id']:<6} {doc['bi_score']:<9.2f} {marker:<10} {doc['text'][:55]}")

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate Stage 1 quality: how good is vector search alone?
# ---------------------------------------------------------------------------
# Let us check the top 5 from vector search (no re-ranking)
top_5_vector = vector_search_results[:5]

relevant_in_top5 = sum(1 for d in top_5_vector if d["truly_relevant"])
total_relevant = sum(1 for d in vector_search_results if d["truly_relevant"])

print("Stage 1 Quality (Vector Search Top 5, NO re-ranking):")
print("=" * 50)
for i, doc in enumerate(top_5_vector):
    marker = "[RELEVANT]" if doc["truly_relevant"] else "[noise]   "
    print(f"  {i+1}. {marker} {doc['doc_id']} (score={doc['bi_score']:.2f}) {doc['text'][:50]}")

print(f"\nPrecision@5 = {relevant_in_top5}/{5} = {relevant_in_top5/5:.2f}")
print(f"Recall@5    = {relevant_in_top5}/{total_relevant} = {relevant_in_top5/total_relevant:.2f}")
print()
print("Problem: D03 and D04 are FALSE POSITIVES.")
print("  D03 is about 'training' not 'serving' -- but 'model' matched.")
print("  D04 is about 'REST APIs' generically -- but 'REST API' matched.")
print("\nThis is exactly why we need re-ranking.")

---

## Step 3: Implement a Cross-Encoder Re-Ranker

A real cross-encoder (like `cross-encoder/ms-marco-MiniLM-L-6-v2`) takes the query and
document text **together** and produces a relevance score. It is more accurate because it
can see the relationship between the query and document at every layer of the model.

Here we simulate a cross-encoder using keyword overlap and semantic heuristics. In
production on Databricks, you would call a Model Serving endpoint running a real
cross-encoder model.

In [ ]:
import re
from typing import List, Dict

# ---------------------------------------------------------------------------
# Simulated cross-encoder re-ranker
# ---------------------------------------------------------------------------
# In production, this would be a model inference call:
#   scores = cross_encoder.predict([(query, doc["text"]) for doc in candidates])
#
# Our simulation uses a scoring heuristic that better captures query intent.

def cross_encoder_score(query: str, document_text: str) -> float:
    """
    Simulate a cross-encoder by scoring query-document relevance.

    A real cross-encoder processes (query, document) jointly through a
    transformer and outputs a single relevance score. Our simulation
    uses weighted keyword matching as an approximation.
    """
    query_lower = query.lower()
    doc_lower = document_text.lower()

    score = 0.0

    # Key concept matching -- does the doc address the core query intent?
    # The real cross-encoder learns these patterns from training data.
    key_concepts = {
        "deploy": 0.15,
        "serving": 0.20,
        "serve": 0.20,
        "endpoint": 0.20,
        "rest api": 0.15,
        "model serving": 0.25,
    }

    for concept, weight in key_concepts.items():
        if concept in doc_lower:
            score += weight

    # Penalize off-topic documents that share surface-level keywords
    off_topic_signals = ["training", "hyperparameter", "tuning", "feature engineering",
                         "data pipeline", "delta lake", "spark cluster"]
    for signal in off_topic_signals:
        if signal in doc_lower and "serving" not in doc_lower and "deploy" not in doc_lower:
            score -= 0.15

    # Normalize to [0, 1]
    return max(0.0, min(1.0, score))


def rerank(query: str, candidates: List[Dict], top_k: int = 5) -> List[Dict]:
    """
    Re-rank candidates using the cross-encoder.

    Steps:
    1. Score each (query, document) pair with the cross-encoder
    2. Sort by cross-encoder score descending
    3. Return the top K
    """
    scored = []
    for doc in candidates:
        ce_score = cross_encoder_score(query, doc["text"])
        scored.append({**doc, "ce_score": ce_score})

    # Sort by cross-encoder score (descending)
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


print("Cross-encoder re-ranker implemented.")
print()
print("How it works (real cross-encoder):")
print("  Input:  (query_text, document_text) pair")
print("  Model:  Transformer that processes both texts jointly")
print("  Output: Single relevance score (0 to 1)")
print()
print("Why it is more accurate than bi-encoder:")
print("  - Bi-encoder: query and doc encoded SEPARATELY, then compared by cosine")
print("  - Cross-encoder: query and doc encoded TOGETHER, model sees interactions")
print("  - Cross-encoder can distinguish 'model training' from 'model serving'")

In [ ]:
# ---------------------------------------------------------------------------
# Run the re-ranker on all 20 candidates
# ---------------------------------------------------------------------------
reranked_results = rerank(query, vector_search_results, top_k=20)

print(f"Re-ranked results (all 20 candidates):")
print(f"{'NewRank':<9} {'DocID':<6} {'BiScore':<9} {'CEScore':<9} {'Relevant':<10} Text")
print("-" * 100)
for i, doc in enumerate(reranked_results):
    marker = "YES" if doc["truly_relevant"] else "---"
    print(f"{i+1:<9} {doc['doc_id']:<6} {doc['bi_score']:<9.2f} {doc['ce_score']:<9.2f} {marker:<10} {doc['text'][:50]}")

---

## Step 4: Compare Before and After Rankings

This is the payoff. Let us put the top 5 from vector search side-by-side with the top 5
after re-ranking and see how the quality improves.

In [ ]:
import pandas as pd

# ---------------------------------------------------------------------------
# Side-by-side comparison: Top 5 before and after re-ranking
# ---------------------------------------------------------------------------
top_5_before = vector_search_results[:5]
top_5_after = rerank(query, vector_search_results, top_k=5)

total_relevant = sum(1 for d in vector_search_results if d["truly_relevant"])

print("BEFORE Re-ranking (Vector Search Top 5)")
print("=" * 70)
relevant_before = 0
for i, doc in enumerate(top_5_before):
    marker = "RELEVANT" if doc["truly_relevant"] else "noise"
    if doc["truly_relevant"]:
        relevant_before += 1
    print(f"  {i+1}. [{marker:<8}] {doc['doc_id']} (bi={doc['bi_score']:.2f}) {doc['text'][:55]}")

print(f"\n  Precision@5 = {relevant_before}/5 = {relevant_before/5:.2f}")
print(f"  Recall@5    = {relevant_before}/{total_relevant} = {relevant_before/total_relevant:.2f}")

print()
print("AFTER Re-ranking (Cross-Encoder Top 5)")
print("=" * 70)
relevant_after = 0
for i, doc in enumerate(top_5_after):
    marker = "RELEVANT" if doc["truly_relevant"] else "noise"
    if doc["truly_relevant"]:
        relevant_after += 1
    print(f"  {i+1}. [{marker:<8}] {doc['doc_id']} (ce={doc['ce_score']:.2f}) {doc['text'][:55]}")

print(f"\n  Precision@5 = {relevant_after}/5 = {relevant_after/5:.2f}")
print(f"  Recall@5    = {relevant_after}/{total_relevant} = {relevant_after/total_relevant:.2f}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary comparison table
# ---------------------------------------------------------------------------
print("\nIMPROVEMENT SUMMARY")
print("=" * 50)

metrics = {
    "Precision@5": (relevant_before / 5, relevant_after / 5),
    "Recall@5": (relevant_before / total_relevant, relevant_after / total_relevant),
}

print(f"{'Metric':<15} {'Before':<10} {'After':<10} {'Change':<10}")
print("-" * 45)
for metric, (before, after) in metrics.items():
    change = after - before
    direction = "+" if change > 0 else ""
    print(f"{metric:<15} {before:<10.2f} {after:<10.2f} {direction}{change:<10.2f}")

print()
print("Key observations:")
print("  - Re-ranking pushed truly relevant docs to the top")
print("  - False positives (D03 about training, D04 about generic REST) moved down")
print("  - The cross-encoder can distinguish 'model serving' from 'model training'")
print("  - This is exactly the kind of nuance bi-encoders miss")

---

## Step 5: When to Use Re-ranking vs When the Latency Is Not Worth It

Re-ranking is not free. Every additional stage adds latency. The decision to use re-ranking
depends on your use case.

In [ ]:
# ---------------------------------------------------------------------------
# Decision framework: when to use re-ranking
# ---------------------------------------------------------------------------
print("WHEN TO USE RE-RANKING")
print("=" * 65)
print()

use_cases = [
    {
        "scenario": "Legal document search",
        "rerank": True,
        "reason": "Missing a relevant precedent could lose the case."
        "  Precision matters more than speed.",
    },
    {
        "scenario": "Medical knowledge base",
        "rerank": True,
        "reason": "Wrong information could harm patients."
        "  Accuracy is non-negotiable.",
    },
    {
        "scenario": "Customer support chatbot",
        "rerank": False,
        "reason": "Users expect fast responses. Good-enough retrieval"
        "  is acceptable if it saves 200ms.",
    },
    {
        "scenario": "Internal knowledge search",
        "rerank": True,
        "reason": "Employees searching for policies need accurate results."
        "  Extra latency is acceptable.",
    },
    {
        "scenario": "Real-time autocomplete",
        "rerank": False,
        "reason": "Must respond in <50ms. Re-ranking is too slow."
        "  Use a fast bi-encoder only.",
    },
    {
        "scenario": "Compliance/audit queries",
        "rerank": True,
        "reason": "Regulatory requirements demand thorough retrieval."
        "  Latency is a secondary concern.",
    },
]

for uc in use_cases:
    decision = "USE re-ranking" if uc["rerank"] else "SKIP re-ranking"
    print(f"  {uc['scenario']}")
    print(f"    Decision: {decision}")
    print(f"    Reason: {uc['reason']}")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Latency comparison
# ---------------------------------------------------------------------------
import time

# Simulate the latency difference
print("LATENCY COMPARISON (typical numbers)")
print("=" * 55)
print()
print(f"{'Stage':<30} {'Latency':<15} {'Notes'}")
print("-" * 65)
print(f"{'Vector Search (bi-encoder)':<30} {'~50ms':<15} {'Pre-indexed, ANN lookup'}")
print(f"{'Re-ranker (cross-encoder)':<30} {'~200ms':<15} {'20 inference calls'}")
print(f"{'LLM generation':<30} {'~1-3s':<15} {'Token-by-token streaming'}")
print("-" * 65)
print(f"{'Total WITHOUT re-ranking':<30} {'~1.1-3.1s':<15} {''}")
print(f"{'Total WITH re-ranking':<30} {'~1.3-3.3s':<15} {''}")
print("-" * 65)
print()
print("Overhead of re-ranking: ~200ms (about 6-18% of total latency)")
print("This is often acceptable because LLM generation dominates the total time.")
print()
print("EXAM RULE OF THUMB:")
print("  If the question mentions precision, accuracy, or quality -> re-ranking helps")
print("  If the question mentions speed, latency, or real-time -> consider skipping")

---

## Exam Tips

| # | Tip | Why It Matters |
|---|-----|----------------|
| 1 | Two-stage retrieval = bi-encoder (fast, approximate) then cross-encoder (slow, precise) | The exam tests the architecture pattern |
| 2 | Cross-encoders process query + document **together**; bi-encoders encode them **separately** | Key architectural distinction |
| 3 | Re-ranking adds latency (~200ms for 20 candidates) | Know the trade-off for scenario-based questions |
| 4 | Use re-ranking when precision > speed (legal, medical, compliance) | Common exam scenario question |
| 5 | Re-ranking does NOT change recall -- it only reorders existing candidates | It cannot find docs that vector search missed |

---

## Key Takeaways

### Architecture
- **Two-stage retrieval**: fast bi-encoder retrieves many candidates, then a slow cross-encoder re-ranks them
- The bi-encoder encodes query and documents **separately** (fast, pre-computable)
- The cross-encoder processes query and document **together** (slow, but more accurate)

### Quality Impact
- Re-ranking pushes truly relevant documents to the top and demotes false positives
- It improves **precision** (fewer irrelevant docs in top K) without changing **recall** (cannot find docs that were not retrieved)
- Especially effective when queries have nuance that embeddings miss (e.g., "training" vs "serving")

### Trade-offs
- Re-ranking adds ~200ms latency for 20 candidates
- Justified for high-stakes use cases (legal, medical, compliance)
- Not justified for real-time / low-latency applications where speed is critical
- The overhead is often small relative to LLM generation time

---

## Next Lab

**Module 03** -- Now that you understand data preparation (chunking, Delta pipelines,
retrieval evaluation, and re-ranking), the next module covers **Model Serving and
Foundation Models** -- deploying and calling models on Databricks.